# NFL Big Data Bowl 2027

Local test build — imports repo modules. Kaggle submission version inlines all code (competition reproducibility rule), assembled Jan 2027.

## Cohort
Combine attempts are linked to supplied game tracking and career outcomes.

In [1]:
import os
from pathlib import Path
from datetime import datetime
from tempfile import TemporaryDirectory
from time import perf_counter
from statistics import correlation
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import display, Markdown
import forty_attempt_features as forty
import player_level as level

root = Path.cwd()
raw = next(path for path in (root / "data", forty.SOURCE) if (path / "combine_tracking.csv").is_file())
figures = Path(os.getenv("NFL_NOTEBOOK_STAGE", str(root))) / "figures_check"
figures.mkdir(exist_ok=True)
results = pl.read_csv(raw / "combine_results.csv", null_values="NA")
official = results.select("nfl_id","draft_year","forty")
players = pl.read_csv(raw / "players.csv", columns=["nfl_id"])
print("Raw source:", raw.relative_to(root))

Raw source: nfl-bdb2027\nfl-big-data-bowl-2027


## Clock method

Both methods project each run onto its first-to-last axis. Method v2 moves the clock origin to the first of three frames above 1.5 yd/s and flags extrapolated finishes. Distance starts at the first frame in both methods.

In [2]:
with TemporaryDirectory() as working:
    attempt_file = Path(working) / "attempts.csv"
    forty.main(source=raw, output=attempt_file)
    attempts = pl.read_csv(attempt_file)
frames = (pl.scan_csv(raw / "combine_tracking.csv", null_values="NA")
    .filter((pl.col("entity_type")=="PLAYER") & (pl.col("drill_type")=="FORTY_YARD_DASH"))
    .select(*forty.ATTEMPT_KEY,"time","x","y","s","a").collect(engine="streaming"))
rows = []
for key, attempt in frames.partition_by(forty.ATTEMPT_KEY, as_dict=True).items():
    ordered = attempt.sort("time").select("time","x","y","s","a")
    start = datetime.fromisoformat(ordered["time"][0])
    samples = [((datetime.fromisoformat(t)-start).total_seconds(), x,y,s,a)
               for t,x,y,s,a in ordered.iter_rows()]
    features = forty.extract_attempt_features(samples, onset_threshold=None, extrapolate=False)
    rows.append(dict(zip(forty.ATTEMPT_KEY,key)) | {"time_40yd_s":features["time_40yd_s"]})
baseline = pl.DataFrame(rows, schema_overrides={"time_40yd_s":pl.Float64})
def validation(df):
    return (df.join(official,on=["nfl_id","draft_year"],validate="m:1")
            .drop_nulls(["time_40yd_s","forty"])
            .with_columns(error=pl.col("time_40yd_s")-pl.col("forty")))
old,new=validation(baseline),validation(attempts)
observed=new.filter(~pl.col("extrapolated"))

Onset threshold: 1.0 yd/s
year,total,crossings_before,crossings_after,mode,n,pearson_r,median_signed_s,median_abs_s
Overall,793,633,789,observed,633,0.937312,0.172281,0.172475
Overall,793,633,789,with_extrapolation,789,0.907404,0.162116,0.163327
2023,262,216,261,observed,216,0.973830,0.132264,0.132264
2023,262,216,261,with_extrapolation,261,0.974702,0.131157,0.131157
2024,269,175,266,observed,175,0.890365,0.181238,0.186221
2024,269,175,266,with_extrapolation,266,0.826061,0.167475,0.170043
2025,262,242,262,observed,242,0.958136,0.206489,0.206489
2025,262,242,262,with_extrapolation,262,0.949221,0.205219,0.205219
Onset threshold: 1.5 yd/s
year,total,crossings_before,crossings_after,mode,n,pearson_r,median_signed_s,median_abs_s
Overall,793,633,789,observed,633,0.957083,0.038989,0.060741
Overall,793,633,789,with_extrapolation,789,0.932016,0.032022,0.059327
2023,262,216,261,observed,216,0.979456,0.010615,0.047692
2023,262,216,261,with_extrapolation,261,0.980825,0.008762,0.045105
2024,269,175

In [3]:
career=pl.read_csv(raw/"player_career_successes.csv",null_values="NA")
plays=(pl.scan_csv(raw/"player_play.csv",null_values="NA").select("nfl_id")
       .drop_nulls().group_by("nfl_id").len(name="play_count").collect(engine="streaming"))
missing_play=players.join(plays,on="nfl_id",how="anti").height
assert missing_play==13
cohort=level.aggregate_players(
    attempts.select("nfl_id","draft_year","event_id","attempt","extrapolated",*level.FEATURES[:-2]),
    results.select("nfl_id","draft_year","combine_position","forty","ten_yd_split"),
    career,plays,
).with_columns(pl.lit(None,dtype=pl.Float64).alias(c) for c in level.SPEED_OUTCOMES)
display(Markdown(level.report(cohort,missing_play)))

# Player-level Spearman correlations

Each cell gives Spearman rho and the pairwise complete player count. Tied values use average ranks. NA denotes too few pairs or a constant variable. Column headers give cohort sizes.

The fastest derived 40-yard attempt supplies the tracking features. Ties use event ID and attempt number. 1/418 players had no timed 40-yard attempt and have blank selected-attempt fields. Games rates use career_games_active and career_games_started divided by 2026 minus draft year. Pro Bowls use the original ballot count and All-Pros sum AP first and second teams.

The supplied player_play.csv has no recorded speed column. Mean and max top speed are missing for all players. Play count uses the number of supplied player-play rows. 13/510 players had no player_play rows in the supplied data. In the 418-player cohort, 11/418 had no rows.

## Primary

| Feature | Outcome | Overall (418) | DB (102) | DL (93) | OL (95) | TE (34) | WR (94) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `time_10yd_s` | `games_started_per_season` | -0.079 (n=417) | -0.165 (n=102) | -0.104 (n=93) | -0.196 (n=94) | 0.012 (n=34) | -0.232 (n=94) |
| `max_s_yd_s` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_40yd_s` | `games_started_per_season` | -0.070 (n=417) | -0.160 (n=102) | -0.115 (n=93) | -0.303 (n=94) | 0.176 (n=34) | -0.268 (n=94) |
| `forty` | `games_started_per_season` | -0.049 (n=418) | -0.122 (n=102) | -0.060 (n=93) | -0.258 (n=95) | 0.152 (n=34) | -0.225 (n=94) |

## Secondary

| Feature | Outcome | Overall (418) | DB (102) | DL (93) | OL (95) | TE (34) | WR (94) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `time_10yd_s` | `games_played_per_season` | -0.055 (n=417) | -0.109 (n=102) | -0.035 (n=93) | -0.103 (n=94) | -0.124 (n=34) | -0.054 (n=94) |
| `time_10yd_s` | `pro_bowls` | -0.036 (n=417) | 0.089 (n=102) | -0.075 (n=93) | -0.174 (n=94) | NA (n=34) | -0.031 (n=94) |
| `time_10yd_s` | `all_pros` | 0.022 (n=417) | 0.164 (n=102) | -0.031 (n=93) | 0.017 (n=94) | NA (n=34) | -0.033 (n=94) |
| `time_10yd_s` | `play_count` | -0.061 (n=406) | -0.215 (n=99) | -0.272 (n=90) | -0.306 (n=93) | -0.181 (n=31) | -0.300 (n=93) |
| `time_10yd_s` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_10yd_s` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_20yd_s` | `games_played_per_season` | -0.069 (n=417) | -0.127 (n=102) | -0.076 (n=93) | -0.162 (n=94) | -0.054 (n=34) | -0.120 (n=94) |
| `time_20yd_s` | `games_started_per_season` | -0.074 (n=417) | -0.149 (n=102) | -0.127 (n=93) | -0.254 (n=94) | 0.146 (n=34) | -0.270 (n=94) |
| `time_20yd_s` | `pro_bowls` | -0.046 (n=417) | 0.113 (n=102) | -0.156 (n=93) | -0.174 (n=94) | NA (n=34) | -0.044 (n=94) |
| `time_20yd_s` | `all_pros` | -0.011 (n=417) | 0.154 (n=102) | -0.062 (n=93) | -0.029 (n=94) | NA (n=34) | -0.046 (n=94) |
| `time_20yd_s` | `play_count` | -0.018 (n=406) | -0.190 (n=99) | -0.273 (n=90) | -0.342 (n=93) | -0.154 (n=31) | -0.280 (n=93) |
| `time_20yd_s` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_20yd_s` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_40yd_s` | `games_played_per_season` | -0.085 (n=417) | -0.151 (n=102) | -0.087 (n=93) | -0.199 (n=94) | -0.008 (n=34) | -0.233 (n=94) |
| `time_40yd_s` | `pro_bowls` | -0.067 (n=417) | 0.050 (n=102) | -0.222 (n=93) | -0.170 (n=94) | NA (n=34) | -0.077 (n=94) |
| `time_40yd_s` | `all_pros` | -0.043 (n=417) | 0.049 (n=102) | -0.082 (n=93) | -0.117 (n=94) | NA (n=34) | -0.079 (n=94) |
| `time_40yd_s` | `play_count` | 0.003 (n=406) | -0.198 (n=99) | -0.240 (n=90) | -0.354 (n=93) | -0.140 (n=31) | -0.229 (n=93) |
| `time_40yd_s` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `time_40yd_s` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `max_s_yd_s` | `games_played_per_season` | 0.089 (n=417) | 0.143 (n=102) | 0.080 (n=93) | 0.206 (n=94) | 0.006 (n=34) | 0.314 (n=94) |
| `max_s_yd_s` | `games_started_per_season` | 0.052 (n=417) | 0.143 (n=102) | 0.059 (n=93) | 0.304 (n=94) | -0.111 (n=34) | 0.145 (n=94) |
| `max_s_yd_s` | `pro_bowls` | 0.097 (n=417) | 0.070 (n=102) | 0.241 (n=93) | 0.063 (n=94) | NA (n=34) | 0.087 (n=94) |
| `max_s_yd_s` | `all_pros` | 0.075 (n=417) | 0.161 (n=102) | 0.074 (n=93) | 0.170 (n=94) | NA (n=34) | 0.087 (n=94) |
| `max_s_yd_s` | `play_count` | -0.042 (n=406) | 0.122 (n=99) | 0.162 (n=90) | 0.284 (n=93) | 0.128 (n=31) | 0.081 (n=93) |
| `max_s_yd_s` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `mean_a_first_0_5s_yd_s2` | `games_played_per_season` | 0.062 (n=417) | 0.152 (n=102) | 0.028 (n=93) | 0.041 (n=94) | 0.240 (n=34) | 0.011 (n=94) |
| `mean_a_first_0_5s_yd_s2` | `games_started_per_season` | 0.125 (n=417) | 0.183 (n=102) | 0.147 (n=93) | 0.180 (n=94) | 0.164 (n=34) | 0.199 (n=94) |
| `mean_a_first_0_5s_yd_s2` | `pro_bowls` | 0.034 (n=417) | -0.055 (n=102) | 0.038 (n=93) | 0.170 (n=94) | NA (n=34) | 0.037 (n=94) |
| `mean_a_first_0_5s_yd_s2` | `all_pros` | -0.010 (n=417) | -0.157 (n=102) | 0.054 (n=93) | 0.025 (n=94) | NA (n=34) | 0.038 (n=94) |
| `mean_a_first_0_5s_yd_s2` | `play_count` | 0.153 (n=406) | 0.268 (n=99) | 0.302 (n=90) | 0.270 (n=93) | 0.257 (n=31) | 0.301 (n=93) |
| `mean_a_first_0_5s_yd_s2` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `mean_a_first_0_5s_yd_s2` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `mean_a_first_1_0s_yd_s2` | `games_played_per_season` | 0.065 (n=417) | 0.166 (n=102) | 0.043 (n=93) | -0.005 (n=94) | 0.143 (n=34) | 0.092 (n=94) |
| `mean_a_first_1_0s_yd_s2` | `games_started_per_season` | 0.118 (n=417) | 0.184 (n=102) | 0.174 (n=93) | 0.130 (n=94) | 0.056 (n=34) | 0.296 (n=94) |
| `mean_a_first_1_0s_yd_s2` | `pro_bowls` | 0.085 (n=417) | -0.043 (n=102) | 0.128 (n=93) | 0.170 (n=94) | NA (n=34) | 0.070 (n=94) |
| `mean_a_first_1_0s_yd_s2` | `all_pros` | 0.039 (n=417) | -0.066 (n=102) | 0.109 (n=93) | 0.032 (n=94) | NA (n=34) | 0.071 (n=94) |
| `mean_a_first_1_0s_yd_s2` | `play_count` | 0.106 (n=406) | 0.286 (n=99) | 0.311 (n=90) | 0.195 (n=93) | 0.241 (n=31) | 0.384 (n=93) |
| `mean_a_first_1_0s_yd_s2` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `mean_a_first_1_0s_yd_s2` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `s_at_40yd_yd_s` | `games_played_per_season` | 0.092 (n=417) | 0.118 (n=102) | 0.097 (n=93) | 0.201 (n=94) | 0.083 (n=34) | 0.334 (n=94) |
| `s_at_40yd_yd_s` | `games_started_per_season` | 0.045 (n=417) | 0.110 (n=102) | 0.068 (n=93) | 0.250 (n=94) | -0.074 (n=34) | 0.138 (n=94) |
| `s_at_40yd_yd_s` | `pro_bowls` | 0.097 (n=417) | 0.070 (n=102) | 0.252 (n=93) | 0.006 (n=94) | NA (n=34) | 0.125 (n=94) |
| `s_at_40yd_yd_s` | `all_pros` | 0.081 (n=417) | 0.161 (n=102) | 0.089 (n=93) | 0.170 (n=94) | NA (n=34) | 0.125 (n=94) |
| `s_at_40yd_yd_s` | `play_count` | -0.043 (n=406) | 0.084 (n=99) | 0.183 (n=90) | 0.243 (n=93) | 0.102 (n=31) | 0.070 (n=93) |
| `s_at_40yd_yd_s` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `s_at_40yd_yd_s` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `forty` | `games_played_per_season` | -0.103 (n=418) | -0.131 (n=102) | -0.094 (n=93) | -0.195 (n=95) | 0.026 (n=34) | -0.428 (n=94) |
| `forty` | `pro_bowls` | -0.101 (n=418) | -0.037 (n=102) | -0.258 (n=93) | -0.070 (n=95) | NA (n=34) | -0.206 (n=94) |
| `forty` | `all_pros` | -0.092 (n=418) | -0.162 (n=102) | -0.126 (n=93) | -0.117 (n=95) | NA (n=34) | -0.207 (n=94) |
| `forty` | `play_count` | 0.042 (n=407) | -0.112 (n=99) | -0.173 (n=90) | -0.220 (n=94) | -0.103 (n=31) | -0.105 (n=93) |
| `forty` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `forty` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `ten_yd_split` | `games_played_per_season` | -0.072 (n=418) | -0.181 (n=102) | -0.131 (n=93) | -0.077 (n=95) | -0.116 (n=34) | -0.223 (n=94) |
| `ten_yd_split` | `games_started_per_season` | -0.052 (n=418) | -0.168 (n=102) | -0.055 (n=93) | -0.154 (n=95) | -0.055 (n=34) | -0.200 (n=94) |
| `ten_yd_split` | `pro_bowls` | -0.042 (n=418) | 0.007 (n=102) | -0.246 (n=93) | -0.121 (n=95) | NA (n=34) | -0.020 (n=94) |
| `ten_yd_split` | `all_pros` | -0.039 (n=418) | -0.087 (n=102) | -0.142 (n=93) | 0.077 (n=95) | NA (n=34) | -0.022 (n=94) |
| `ten_yd_split` | `play_count` | 0.063 (n=407) | -0.200 (n=99) | -0.164 (n=90) | -0.106 (n=94) | 0.029 (n=31) | -0.088 (n=93) |
| `ten_yd_split` | `player_play_mean_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |
| `ten_yd_split` | `player_play_max_top_speed` | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) | NA (n=0) |


## Speed transfer

Game speed uses the maximum and 99th percentile of supplied player frames, pooled across seasons.

In [4]:
engine="CPU / Polars streaming"
t0=perf_counter()
scans=[pl.scan_csv(raw/f"game_tracking_{y}.csv",null_values="NA",
                   schema_overrides={"nfl_id":pl.Int64,"s":pl.Float64})
       .select("nfl_id","s","game_id","play_id") for y in (2023,2024,2025)]
game=(pl.concat(scans).join(players.lazy(),on="nfl_id",how="semi").group_by("nfl_id")
      .agg(pl.col("s").max().alias("game_max_s"),
           pl.col("s").quantile(.99,interpolation="linear").alias("game_p99_s"),
           pl.struct("game_id","play_id").n_unique().alias("game_plays"),
           pl.col("game_id").n_unique().alias("game_games")).collect(engine="streaming"))
print(f"GAME_STAGE_SECONDS={perf_counter()-t0:.3f} ({engine})")
cohort=cohort.join(game,on="nfl_id",how="left",validate="1:1").with_columns(
    plays_per_season=pl.col("game_plays")/pl.col("seasons"))
def rho(df,x,y):
    pair=df.select(x,y).drop_nulls()
    return correlation(pair[x].rank(method="average"),pair[y].rank(method="average")),pair.height
positions=["DB","DL","OL","TE","WR"]
groups={k:cohort.filter(pl.col("combine_position")==k) for k in positions}
speed_cols=["game_max_s","game_p99_s"]
speed={k:[rho(g,"max_s_yd_s",c) for c in speed_cols] for k,g in {"Pooled":cohort,**groups}.items()}

GAME_STAGE_SECONDS=3.334 (CPU / Polars streaming)


## Acceleration

Starts per season use career starts divided by seasons since the draft. Sensitivity excludes selected extrapolated crossings. The official split correlation is sign-adjusted so that faster corresponds to a higher value.

In [5]:
accel,starts="mean_a_first_0_5s_yd_s2","games_started_per_season"
def exclude(frame):
    return frame.filter(~pl.col("extrapolated").fill_null(False))
sensitivity=exclude(cohort)

## Limitations

These correlations are descriptive. Thirteen of 510 players have no supplied player-play rows. A missing join indicates missing supplied tracking. Multiple attempts can share one official time.

In [6]:
teal,amber,slate="#087E83","#B45309","#53647a"
colors=["#2366A8",teal,"#8054A1",amber,"#AD425F"]
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,
    "axes.axisbelow":True,"font.size":10,"legend.frameon":False})
def panels(title,width=7.6,count=1,**options):
    fig,axes=plt.subplots(1,count,figsize=(width,5.3),**options)
    fig.subplots_adjust(left=.12 if count==1 else .09,right=.96,bottom=.22,top=.80,wspace=.31)
    fig.text(fig.subplotpars.left,.91,title,fontsize=15,fontweight="bold")
    return fig,axes
def save(fig,name,caption):
    for ax in fig.axes:
        ax.grid(axis="y",color="#dee5ed",linewidth=.7)
    fig.text(fig.subplotpars.left,.055,caption,fontsize=9,color=slate)
    fig.savefig(figures/name,dpi=200)
    plt.close(fig)
fig,ax=panels("Method v2 forty times track official times")
lo=min(new["forty"].min(),new["time_40yd_s"].min())-.04
hi=max(new["forty"].max(),new["time_40yd_s"].max())+.04
ax.plot([lo,hi],[lo,hi],"--",color=slate,label="Identity")
for flag,mark,color,label in [(False,"o",teal,"Observed"),(True,"^",amber,"Extrapolated")]:
    q=new.filter(pl.col("extrapolated")==flag)
    ax.scatter(q["forty"],q["time_40yd_s"],s=17,marker=mark,color=color,alpha=.55,linewidths=0,label=label)
ax.set(xlim=(lo,hi),ylim=(lo,hi),xlabel="Official forty (s)",ylabel="Derived forty (s)")
ax.set_aspect("equal")
ax.legend(loc="upper left")
save(fig,"fig1_validation_scatter.png",f"n={new.height} attempts | {observed.height} observed, {new.height-observed.height} extrapolated")
fig,axes=panels("Method v2 reduces median signed forty error",10,2,sharex=True,sharey=True)
bins=np.histogram_bin_edges(np.concatenate([old["error"].to_numpy(),new["error"].to_numpy()]),bins=36)
for ax,data,title,color in zip(axes,[old,new],["Frozen first frame","Method v2"],[colors[0],teal]):
    med=data["error"].median()
    ax.hist(data["error"].to_numpy(),bins=bins,color=color,edgecolor="white",linewidth=.6)
    ax.axvline(0,color=slate,ls=":")
    ax.axvline(med,color=amber,ls="--")
    ax.text(.97,.95,f"Median {med:+.3f} s",transform=ax.transAxes,ha="right",va="top",color=amber)
    ax.set_title(title,loc="left")
    ax.set_xlabel("Derived minus official time (s)")
axes[0].set_ylabel("Attempts (count)")
save(fig,"fig2_error_fix.png",f"First frame n={old.height} | Method v2 n={new.height} | Shared axes and bins")

fig,ax=panels("Combine max speed and game p99 speed")
for (k,g),color in zip(groups.items(),colors):
    q=g.select("max_s_yd_s","game_p99_s").drop_nulls()
    ax.scatter(q["max_s_yd_s"],q["game_p99_s"],s=23,color=color,alpha=.72,linewidths=0,label=k)
ax.set(xlabel="Combine max speed (yd/s)",ylabel="Game p99 speed (yd/s)")
ax.legend(title="Combine position",loc="lower right",ncol=2)
save(fig,"fig3_speed_transfer_scatter.png",f"n={speed['Pooled'][1][1]} players | Pooled rho {speed['Pooled'][1][0]:.3f}")

def bars(title,width,series,labels,filename,caption,limits,references=None):
    fig,ax=panels(title,width)
    index=np.arange(len(positions))
    for i,(label,color) in enumerate(zip(labels,[teal,amber])):
        ax.bar(index+(i-.5)*.32,[r for r,n in series[i]],.30,label=label,color=color)
        if references:
            ax.axhline(references[i],ls="--",color=color,label=f"Pooled {label} ({references[i]:.3f})")
    ax.axhline(0,color=slate,lw=.9)
    ax.set(xticks=index,xticklabels=positions,xlabel="Combine position",ylabel="Spearman rho (unitless)",ylim=limits)
    ax.legend(ncol=2,loc="upper left",fontsize=8)
    save(fig,filename,caption)
bars("Combine-to-game speed correlations vary by position",8.4,
     [[speed[k][i] for k in positions] for i in range(2)],["Game max","Game p99"],
     "fig4_speed_transfer_bars.png","Pairwise n per metric | "+", ".join(f"{k} {speed[k][0][1]}" for k in positions),
     (0,1.03),[r for r,n in speed["Pooled"]])
acceleration=[[rho(g,accel,starts) for g in groups.values()],
              [rho(exclude(g),accel,starts) for g in groups.values()]]
bars("Early acceleration and starts have positive rho in all groups",9.7,acceleration,
     ["Full","Excluding extrapolated"],"fig5_accel_starts_bars.png",
     "Pairwise n full / excluding extrapolated | "+", ".join(f"{k} {acceleration[0][i][1]}/{acceleration[1][i][1]}" for i,k in enumerate(positions)),
     (-.026,.27))
fig,axes=panels("Forty time vs. plays and first-half-second acceleration vs. starts",11.5,2)
ns=[]
for ax,x,y,xl,yl in [
    (axes[0],"time_40yd_s","play_count","Derived forty time (s)","Supplied player-play rows (count)"),
    (axes[1],accel,starts,"Mean acceleration, 0-0.5 s (yd/s\u00b2)","Games started per season (starts/season)")]:
    q=cohort.select(x,y).drop_nulls()
    r,n=rho(cohort,x,y)
    ns.append(n)
    ax.scatter(q[x],q[y],s=19,alpha=.48,color=teal,linewidths=0)
    ax.set(xlabel=xl,ylabel=yl)
    ax.text(.98,.96,f"rho {r:+.3f} | n={n}",transform=ax.transAxes,ha="right",va="top")
save(fig,"fig6_first_half_second.png",f"Left n={ns[0]} players | Right n={ns[1]} players")
classes=cohort.group_by("draft_year").agg(pl.len(),pl.col("game_games").count(),pl.col("extrapolated").sum()).sort("draft_year")
lines=["# Cohort and tracking coverage","","| Draft class | Tracked n | Game-tracking n | Extrapolated n |",
       "| --- | ---: | ---: | ---: |",*[f"| {y} | {n} | {g} | {e} |" for y,n,g,e in classes.rows()],
       f"| **Total** | **{cohort.height}** | **{cohort['game_games'].count()}** | **{cohort['extrapolated'].sum()}** |","",
       "Tracked n counts players with combine forty tracking. Game-tracking n counts players with supplied game tracking across the 2023\u20132025 seasons. Extrapolated n counts selected attempts flagged as extrapolated.",""]
(figures/"table1_cohort.md").write_text("\n".join(lines),encoding="utf-8")

454

In [7]:
checks=[]
def add(label,v,t,tol=0,backend="CPU"):
    checks.append((label,v,t,tol,backend,abs(v-t)<=tol))
for name,df,n,r,mae in [("Method v2",new,789,.932,.0593),("Observed v2",observed,633,.9571,.0607),("Baseline",old,633,.889,.356)]:
    add(name+" n",df.height,n)
    add(name+" Pearson r",correlation(df["forty"],df["time_40yd_s"]),r,.002)
    add(name+" median abs error (s)",df["error"].abs().median(),mae,.002)
add("Baseline signed median (s)",old["error"].median(),.356,.002)
add("Tracked players",cohort["nfl_id"].n_unique(),418)
add("Attempts",attempts.height,793)
add("Full players",players.height,510)
add("Game join full",game.height,497,backend=engine)
add("Game join tracked",cohort["game_games"].count(),407,backend=engine)
for group,targets in {"Pooled":(.810,.850),"DL":(.652,.682),"DB":(.285,.289),"WR":(.430,.539)}.items():
    for col,(r,n),target in zip(speed_cols,speed[group],targets):
        add(f"{group} {col} rho",r,target,.005)
        if group=="Pooled":
            add(f"{group} {col} n",n,406)
for col,target in [("time_10yd_s",-.079),("time_40yd_s",-.070),("forty",-.049)]:
    add(f"{col} vs starts rho",rho(cohort,col,starts)[0],target,.005)
for label,g,a,s in [("Full",cohort,.1252,.0518),("Excluding extrapolated",sensitivity,.0877,.0188)]:
    add(label+" acceleration rho",rho(g,accel,starts)[0],a,.005)
    add(label+" sign-adjusted split rho",-rho(g,"ten_yd_split",starts)[0],s,.005)
add("Forty vs play count rho",rho(cohort,"time_40yd_s","play_count")[0],.003,.005)
expected={"fig1_validation_scatter.png","fig2_error_fix.png","fig3_speed_transfer_scatter.png",
          "fig4_speed_transfer_bars.png","fig5_accel_starts_bars.png","fig6_first_half_second.png","table1_cohort.md"}
present={f.name for f in figures.iterdir()}
add("Expected files",len(expected & present),7)
for name in sorted(expected):
    if name.endswith(".png") and name in present:
        with Image.open(figures/name) as im:
            dpi=im.info.get("dpi",(0,0))
            im.verify()
        for axis,value in zip("xy",dpi):
            add(name+" "+axis+" dpi",round(value,3),200)
table=["| Check | Recomputed | Frozen | Tolerance | Engine | Result |","|---|---:|---:|---:|---|---|"]
table += [f"| {k} | {v:.7g} | {t:.7g} | {tol:g} | {e} | {'PASS' if ok else 'FAIL'} |"
          for k,v,t,tol,e,ok in checks]
display(Markdown("\n".join(table)))
print("\n".join(table))
assert all(check[-1] for check in checks), [(k,v,t,v-t,e) for k,v,t,tol,e,ok in checks if not ok]
for name in sorted(expected):
    print("SIZE",name,(root/"figures"/name).stat().st_size,(figures/name).stat().st_size)

| Check | Recomputed | Frozen | Tolerance | Engine | Result |
|---|---:|---:|---:|---|---|
| Method v2 n | 789 | 789 | 0 | CPU | PASS |
| Method v2 Pearson r | 0.932016 | 0.932 | 0.002 | CPU | PASS |
| Method v2 median abs error (s) | 0.05932737 | 0.0593 | 0.002 | CPU | PASS |
| Observed v2 n | 633 | 633 | 0 | CPU | PASS |
| Observed v2 Pearson r | 0.9570828 | 0.9571 | 0.002 | CPU | PASS |
| Observed v2 median abs error (s) | 0.06074138 | 0.0607 | 0.002 | CPU | PASS |
| Baseline n | 633 | 633 | 0 | CPU | PASS |
| Baseline Pearson r | 0.8889394 | 0.889 | 0.002 | CPU | PASS |
| Baseline median abs error (s) | 0.3558334 | 0.356 | 0.002 | CPU | PASS |
| Baseline signed median (s) | 0.3558334 | 0.356 | 0.002 | CPU | PASS |
| Tracked players | 418 | 418 | 0 | CPU | PASS |
| Attempts | 793 | 793 | 0 | CPU | PASS |
| Full players | 510 | 510 | 0 | CPU | PASS |
| Game join full | 497 | 497 | 0 | CPU / Polars streaming | PASS |
| Game join tracked | 407 | 407 | 0 | CPU / Polars streaming | PASS |
| Pooled game_max_s rho | 0.8104335 | 0.81 | 0.005 | CPU | PASS |
| Pooled game_max_s n | 406 | 406 | 0 | CPU | PASS |
| Pooled game_p99_s rho | 0.849877 | 0.85 | 0.005 | CPU | PASS |
| Pooled game_p99_s n | 406 | 406 | 0 | CPU | PASS |
| DL game_max_s rho | 0.6518547 | 0.652 | 0.005 | CPU | PASS |
| DL game_p99_s rho | 0.6822977 | 0.682 | 0.005 | CPU | PASS |
| DB game_max_s rho | 0.2850229 | 0.285 | 0.005 | CPU | PASS |
| DB game_p99_s rho | 0.289014 | 0.289 | 0.005 | CPU | PASS |
| WR game_max_s rho | 0.4295494 | 0.43 | 0.005 | CPU | PASS |
| WR game_p99_s rho | 0.5392429 | 0.539 | 0.005 | CPU | PASS |
| time_10yd_s vs starts rho | -0.07923206 | -0.079 | 0.005 | CPU | PASS |
| time_40yd_s vs starts rho | -0.06961875 | -0.07 | 0.005 | CPU | PASS |
| forty vs starts rho | -0.04851314 | -0.049 | 0.005 | CPU | PASS |
| Full acceleration rho | 0.1252173 | 0.1252 | 0.005 | CPU | PASS |
| Full sign-adjusted split rho | 0.05177271 | 0.0518 | 0.005 | CPU | PASS |
| Excluding extrapolated acceleration rho | 0.08765478 | 0.0877 | 0.005 | CPU | PASS |
| Excluding extrapolated sign-adjusted split rho | 0.01876998 | 0.0188 | 0.005 | CPU | PASS |
| Forty vs play count rho | 0.003330976 | 0.003 | 0.005 | CPU | PASS |
| Expected files | 7 | 7 | 0 | CPU | PASS |
| fig1_validation_scatter.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig1_validation_scatter.png y dpi | 200 | 200 | 0 | CPU | PASS |
| fig2_error_fix.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig2_error_fix.png y dpi | 200 | 200 | 0 | CPU | PASS |
| fig3_speed_transfer_scatter.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig3_speed_transfer_scatter.png y dpi | 200 | 200 | 0 | CPU | PASS |
| fig4_speed_transfer_bars.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig4_speed_transfer_bars.png y dpi | 200 | 200 | 0 | CPU | PASS |
| fig5_accel_starts_bars.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig5_accel_starts_bars.png y dpi | 200 | 200 | 0 | CPU | PASS |
| fig6_first_half_second.png x dpi | 200 | 200 | 0 | CPU | PASS |
| fig6_first_half_second.png y dpi | 200 | 200 | 0 | CPU | PASS |

| Check | Recomputed | Frozen | Tolerance | Engine | Result |
|---|---:|---:|---:|---|---|
| Method v2 n | 789 | 789 | 0 | CPU | PASS |
| Method v2 Pearson r | 0.932016 | 0.932 | 0.002 | CPU | PASS |
| Method v2 median abs error (s) | 0.05932737 | 0.0593 | 0.002 | CPU | PASS |
| Observed v2 n | 633 | 633 | 0 | CPU | PASS |
| Observed v2 Pearson r | 0.9570828 | 0.9571 | 0.002 | CPU | PASS |
| Observed v2 median abs error (s) | 0.06074138 | 0.0607 | 0.002 | CPU | PASS |
| Baseline n | 633 | 633 | 0 | CPU | PASS |
| Baseline Pearson r | 0.8889394 | 0.889 | 0.002 | CPU | PASS |
| Baseline median abs error (s) | 0.3558334 | 0.356 | 0.002 | CPU | PASS |
| Baseline signed median (s) | 0.3558334 | 0.356 | 0.002 | CPU | PASS |
| Tracked players | 418 | 418 | 0 | CPU | PASS |
| Attempts | 793 | 793 | 0 | CPU | PASS |
| Full players | 510 | 510 | 0 | CPU | PASS |
| Game join full | 497 | 497 | 0 | CPU / Polars streaming | PASS |
| Game join tracked | 407 | 407 | 0 | CPU / Polars streaming | PASS 